# 예제 02. 반복 실행은 독립이 아니다 — 군집 보정

전산실험 · AI Agent 만들기 · **11주차** · 예상 소요 40분

## 목표
- 반복 실행 자료가 **군집자료**임을 이해한다
- naive 신뢰구간이 **왜 좁은지** 시뮬레이션으로 확인한다
- ICC 와 설계효과를 계산한다
- **문항을 늘릴지 반복을 늘릴지** 계산으로 정한다

> ★ **이 노트북이 이 수업 전체에서 가장 중요한 것 중 하나입니다.** ★
> 대부분의 AI 평가 코드가 여기서 틀립니다.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile evaluate.py
"""전산실험2 11주차 : 에이전트 평가 — 성능을 추정 문제로 다룬다.

지금까지 매주 성공률을 재기는 했다. 그런데 n=10, n=20 수준이었고
신뢰구간이 너무 넓어 사실 아무것도 주장할 수 없었다.

이 모듈은 그것을 제대로 한다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 수업이 다른 AI 강의와 갈리는 지점

 에이전트를 k번 반복 실행해 얻은 관측은 **서로 독립이 아니다.**
 같은 문항을 여러 번 돌린 것이기 때문이다. 어떤 문항은 늘 맞고
 어떤 문항은 늘 틀린다. 문항 안의 관측은 서로 닮아 있다.

 그런데 대부분의 AI 평가 코드는 이것을 무시하고
 (맞은 횟수) / (전체 시행) 을 그냥 이항비율로 다룬다.
 → **신뢰구간이 실제보다 좁게 나온다. 없는 차이를 있다고 말하게 된다.**

 이것은 통계학에서 오래 알려진 **군집자료(clustered data)** 문제다.
 여러분은 이것을 아는 사람들이다. 제대로 하자.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

구성:
    GoldenItem / GoldenSet   정답이 정해진 평가 문항
    채점 함수들               keyword / numeric / choice / judge
    run_eval                 반복 실행해 결과를 모은다
    summarize                정확도 + 신뢰구간 (군집 보정 포함)
    compare_conditions       두 조건을 짝지어 비교 (McNemar)
    required_n               표본 크기 계산

사용 예:
    from evaluate import GoldenItem, run_eval, summarize

    items = [GoldenItem("q1", "치료 전후 차이가 있나?", expected="유의",
                        scorer="keyword")]
    df = run_eval(items, lambda q: agent.run(q).final_answer, repeats=5)
    print(summarize(df))
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = [
    "GoldenItem",
    "score_keyword",
    "score_numeric",
    "score_choice",
    "make_judge",
    "run_eval",
    "summarize",
    "per_item",
    "consistency",
    "compare_conditions",
    "required_n",
    "wilson",
]


# --------------------------------------------------------------------------
# 평가 문항
# --------------------------------------------------------------------------
@dataclass
class GoldenItem:
    """정답이 정해진 평가 문항 하나.

    Parameters
    ----------
    id       문항 식별자
    question 에이전트에게 줄 질문
    expected 정답. 채점 방식에 따라 형태가 다르다
             keyword -> str 또는 list[str] (모두 포함되어야 함)
             numeric -> float
             choice  -> str (허용 목록 중 하나)
    scorer   "keyword" | "numeric" | "choice" | callable
    tolerance numeric 채점의 상대 허용오차
    tags     분석용 꼬리표 (난이도, 유형 등)
    """

    id: str
    question: str
    expected: Any
    scorer: Any = "keyword"
    tolerance: float = 0.01
    tags: dict[str, Any] = field(default_factory=dict)


# --------------------------------------------------------------------------
# 채점 함수
# --------------------------------------------------------------------------
def score_keyword(answer: str, expected: Any, **_: Any) -> bool:
    """정답 키워드가 모두 들어 있는가.

    가장 단순하고 가장 자주 쓴다. 다만 **표현이 달라지면 놓친다.**
    "유의하다" 를 기대했는데 "통계적으로 의미 있는 차이" 라고 답하면 오답 처리된다.
    이 한계를 알고 써야 한다. 대안은 `make_judge`.
    """
    if answer is None:
        return False
    본문 = str(answer).lower()
    필요 = [expected] if isinstance(expected, str) else list(expected)
    return all(str(k).lower() in 본문 for k in 필요)


_NUM = re.compile(r"[-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?")


def score_numeric(answer: str, expected: float, tolerance: float = 0.01, **_: Any) -> bool:
    """답변에 들어 있는 수치 중 하나가 정답과 충분히 가까운가.

    p값처럼 자릿수가 큰 값은 **상대오차**로 봐야 한다.
    p=1e-20 과 p=2e-20 의 절대차는 0에 가깝지만 둘은 다른 값이 아니다
    (둘 다 "매우 작다"). 반대로 0.04 와 0.06 은 절대차가 작아도
    결론이 갈린다. 여기서는 상대오차를 쓰되, 문항에서 tolerance 를 조절한다.
    """
    if answer is None:
        return False
    후보 = []
    for m in _NUM.findall(str(answer)):
        try:
            후보.append(float(m.replace(",", "")))
        except ValueError:
            pass
    if not 후보:
        return False
    if expected == 0:
        return any(abs(v) <= tolerance for v in 후보)
    return any(abs(v - expected) / abs(expected) <= tolerance for v in 후보)


def score_choice(answer: str, expected: str, **_: Any) -> bool:
    """답변이 정답 선택지를 명시했는가. 구두점·대소문자·공백 차이를 무시한다."""
    if answer is None:
        return False
    정규 = lambda s: re.sub(r"[\s\-_]", "", str(s).lower())
    return 정규(expected) in 정규(answer)


def make_judge(llm: Any, criteria: str = "", **kw: Any) -> Callable[..., bool]:
    """LLM-as-judge 채점기를 만든다.

    표현이 달라도 뜻이 맞으면 정답으로 인정하고 싶을 때 쓴다.

    **주의: 판정자도 모델이다.**
      - 판정자가 틀릴 수 있다 → 사람이 채점한 것과 일치율을 먼저 재라
      - 판정자와 피평가자가 같은 모델이면 편향이 생긴다 (자기 답을 후하게 봄)
      - 판정 자체가 확률적이다 → 판정도 여러 번 해서 다수결

    그래서 이 수업의 기본 채점은 keyword/numeric 이고,
    judge 는 **그것으로 안 되는 문항에만** 쓴다.
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 200)

    SYSTEM = (
        "너는 채점자다. 학생의 답변이 정답과 같은 뜻인지 판정한다.\n"
        "표현이 달라도 내용이 맞으면 정답으로 인정한다.\n"
        "핵심 결론이 다르면 오답이다. 애매하면 오답으로 처리한다.\n"
        '반드시 {"correct": true} 또는 {"correct": false} 형태의 JSON만 출력한다.'
        + (f"\n추가 기준: {criteria}" if criteria else "")
    )

    def judge(answer: str, expected: Any, **_: Any) -> bool:
        from structured import JSONParseError, parse_json

        if answer is None:
            return False
        prompt = (f"## 정답\n{expected}\n\n## 학생 답변\n{answer}\n\n"
                  f"같은 뜻인가? JSON만 출력하라.")
        try:
            out = llm.chat([{"role": "system", "content": SYSTEM},
                            {"role": "user", "content": prompt}], **kw)
            return bool(parse_json(out).get("correct", False))
        except (JSONParseError, Exception):
            return False

    return judge


_SCORERS = {"keyword": score_keyword, "numeric": score_numeric, "choice": score_choice}


def _resolve(scorer: Any) -> Callable[..., bool]:
    if callable(scorer):
        return scorer
    if scorer in _SCORERS:
        return _SCORERS[scorer]
    raise ValueError(f"알 수 없는 채점 방식: {scorer!r}. 가능: {list(_SCORERS)} 또는 함수")


# --------------------------------------------------------------------------
# 실행
# --------------------------------------------------------------------------
def run_eval(
    items: list[GoldenItem],
    run_fn: Callable[[str], Any],
    *,
    repeats: int = 5,
    condition: str = "base",
    verbose: bool = True,
    on_error: str = "wrong",
) -> Any:
    """각 문항을 `repeats` 회 실행하고 채점한다.

    Parameters
    ----------
    run_fn : 질문을 받아 답변(문자열 또는 객체)을 돌려주는 함수
    repeats : 문항당 반복 횟수. **1이면 안 된다.** 변동을 못 본다
    on_error : 실행이 예외로 죽었을 때 "wrong"(오답 처리) 또는 "skip"

    Returns
    -------
    DataFrame : item_id, rep, condition, correct, answer, elapsed, error
                **long format** — 한 행이 한 번의 시행
    """
    import time

    import pandas as pd

    행 = []
    for item in items:
        점수함수 = _resolve(item.scorer)
        for rep in range(1, repeats + 1):
            t0 = time.perf_counter()
            answer, err = None, None
            try:
                answer = run_fn(item.question)
            except Exception as exc:
                err = f"{type(exc).__name__}: {exc}"
                if on_error == "skip":
                    continue
            경과 = time.perf_counter() - t0

            correct = False
            if err is None:
                try:
                    correct = bool(점수함수(answer, item.expected,
                                          tolerance=item.tolerance))
                except Exception as exc:
                    err = f"채점 실패: {type(exc).__name__}: {exc}"

            행.append({
                "item_id": item.id, "rep": rep, "condition": condition,
                "correct": correct, "answer": str(answer)[:500],
                "elapsed": round(경과, 2), "error": err,
                **{f"tag_{k}": v for k, v in item.tags.items()},
            })
            if verbose:
                print("O" if correct else ("E" if err else "X"), end="", flush=True)
        if verbose:
            print(f"  {item.id}")
    return pd.DataFrame(행)


# --------------------------------------------------------------------------
# 집계
# --------------------------------------------------------------------------
def wilson(successes: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """이항비율의 Wilson 신뢰구간.

    n이 작거나 비율이 0/1에 가까울 때 정규근사(Wald)보다 훨씬 낫다.
    Wald 는 [0,1] 을 벗어나거나 폭이 0이 되어 버린다.
    """
    if n == 0:
        return (0.0, 1.0)
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def per_item(df: Any) -> Any:
    """문항별 정확도. **어떤 문항이 어려운지 보는 것이 지표 하나보다 중요하다.**"""
    import pandas as pd

    g = df.groupby("item_id", sort=False)["correct"]
    out = pd.DataFrame({"n": g.count(), "맞음": g.sum(), "정확도": g.mean()})
    out["항상맞음"] = out["정확도"] == 1.0
    out["항상틀림"] = out["정확도"] == 0.0
    return out.reset_index()


def summarize(df: Any, alpha: float = 0.05) -> dict[str, Any]:
    """정확도를 **군집 보정과 함께** 요약한다.

    ★ 이 함수가 이 모듈의 핵심이다. ★

    같은 문항을 반복 실행한 관측은 서로 독립이 아니다.
    그래서 두 가지를 함께 보고한다.

      naive_ci   : 모든 시행을 독립으로 본 Wilson 구간 (대부분의 AI 평가 코드)
      cluster_ci : 문항을 군집으로 본 구간 (문항별 정확도의 표본평균 기반)

    `design_effect` 가 1보다 크면 naive 구간이 **실제보다 좁다**는 뜻이다.
    즉 없는 차이를 있다고 말할 위험이 있다.
    """
    import numpy as np
    from scipy.stats import t as t_dist

    n_trials = len(df)
    n_correct = int(df["correct"].sum())
    p_hat = n_correct / n_trials if n_trials else 0.0

    # (1) 순진한 구간 — 모든 시행이 독립이라고 가정
    naive = wilson(n_correct, n_trials, alpha)

    # (2) 군집 보정 — 문항별 정확도를 하나의 관측으로 본다
    문항정확도 = df.groupby("item_id", sort=False)["correct"].mean().values
    m = len(문항정확도)
    p_cluster = float(문항정확도.mean()) if m else 0.0
    if m > 1:
        se = float(문항정확도.std(ddof=1) / math.sqrt(m))
        tcrit = t_dist.ppf(1 - alpha / 2, df=m - 1)
        cluster = (max(0.0, p_cluster - tcrit * se), min(1.0, p_cluster + tcrit * se))
    else:
        se, cluster = float("nan"), (0.0, 1.0)

    # (3) 급내상관 ICC — 같은 문항 안의 관측이 얼마나 닮았는가
    #     일원 임의효과 모형의 표준 추정량을 쓴다.
    #        ICC = (MSB - MSW) / (MSB + (k-1) MSW)
    #     신뢰구간 폭의 비율로 역산하면 t분포 임계값 때문에 1을 넘어 버린다.
    icc = _icc_oneway(df)

    # (4) 설계효과 — 표준 공식. 실효표본 = 전체시행 / deff
    k_bar = n_trials / m if m else 0
    deff = 1 + (k_bar - 1) * icc if not math.isnan(icc) else float("nan")

    naive_width = naive[1] - naive[0]
    cluster_width = cluster[1] - cluster[0]

    return {
        "n_items": m,
        "n_trials": n_trials,
        "reps_per_item": round(k_bar, 1),
        "accuracy": round(p_hat, 4),
        "accuracy_by_item": round(p_cluster, 4),
        "naive_ci": (round(naive[0], 4), round(naive[1], 4)),
        "cluster_ci": (round(cluster[0], 4), round(cluster[1], 4)),
        "cluster_se": round(se, 4) if not math.isnan(se) else None,
        "icc": round(icc, 3) if not math.isnan(icc) else None,
        "design_effect": round(deff, 2) if not math.isnan(deff) else None,
        "effective_n": round(n_trials / deff, 1) if deff and deff > 0 else None,
        "ci_width_ratio": round(cluster_width / naive_width, 2) if naive_width > 0 else None,
        "n_errors": int(df["error"].notna().sum()),
    }


def _icc_oneway(df: Any) -> float:
    """일원 임의효과 급내상관.

        ICC = (MSB - MSW) / (MSB + (k-1) MSW)

    문항이 군집, 반복 시행이 군집 내 관측이다.
    음수가 나오면 0으로 자른다 (군집 효과가 없다는 뜻).
    """
    그룹 = [g["correct"].astype(float).values for _, g in df.groupby("item_id", sort=False)]
    m = len(그룹)
    if m < 2:
        return float("nan")
    n_i = [len(g) for g in 그룹]
    if min(n_i) < 2:
        return float("nan")

    전체평균 = sum(g.sum() for g in 그룹) / sum(n_i)
    ss_b = sum(len(g) * (g.mean() - 전체평균) ** 2 for g in 그룹)
    ss_w = sum(((g - g.mean()) ** 2).sum() for g in 그룹)
    df_b, df_w = m - 1, sum(n_i) - m
    if df_w <= 0:
        return float("nan")
    ms_b, ms_w = ss_b / df_b, ss_w / df_w

    # 균형이 안 맞을 때의 유효 군집 크기
    N = sum(n_i)
    k0 = (N - sum(x * x for x in n_i) / N) / (m - 1)
    if k0 <= 0 or (ms_b + (k0 - 1) * ms_w) == 0:
        return float("nan")
    icc = (ms_b - ms_w) / (ms_b + (k0 - 1) * ms_w)
    return float(min(1.0, max(0.0, icc)))


def consistency(df: Any) -> dict[str, Any]:
    """같은 문항을 반복했을 때 얼마나 일관되는가.

    정확도가 같아도 성질이 전혀 다를 수 있다.

      A: 문항의 절반은 늘 맞고 절반은 늘 틀림  → 정확도 50%, 일관성 100%
      B: 모든 문항을 반반씩 맞힘              → 정확도 50%, 일관성 50%

    A는 "못 푸는 문제가 정해져 있다" 이고 B는 "매번 운이다" 이다.
    **고쳐야 할 것이 완전히 다르다.**
    """
    import numpy as np

    문항 = df.groupby("item_id", sort=False)["correct"]
    정확도 = 문항.mean()
    # 다수결과 일치하는 비율 (0.5 -> 0.5, 0 또는 1 -> 1.0)
    다수결일치 = 정확도.apply(lambda p: max(p, 1 - p))
    return {
        "항상 맞음": int((정확도 == 1.0).sum()),
        "항상 틀림": int((정확도 == 0.0).sum()),
        "들쭉날쭉": int(((정확도 > 0) & (정확도 < 1)).sum()),
        "평균 다수결 일치율": round(float(다수결일치.mean()), 3),
        "문항 간 정확도 표준편차": round(float(정확도.std(ddof=1)), 3) if len(정확도) > 1 else None,
    }


# --------------------------------------------------------------------------
# 조건 비교
# --------------------------------------------------------------------------
def compare_conditions(
    df_a: Any, df_b: Any, *, name_a: str = "A", name_b: str = "B", alpha: float = 0.05
) -> dict[str, Any]:
    """두 조건(프롬프트 A/B 등)을 비교한다.

    **같은 문항으로 두 조건을 돌렸으므로 짝지은 자료다.**
    독립 두 표본 검정을 쓰면 문항 난이도 차이가 오차에 섞여 검정력을 잃는다.
    4주차에 "대응표본을 독립표본으로 다루지 마라" 고 한 것과 같은 이야기다.

    두 가지 방법을 함께 준다.
      - 문항별 다수결을 이분값으로 만들어 **McNemar 검정**
      - 문항별 정확도 차이에 **대응표본 t / Wilcoxon**
    """
    import pandas as pd
    from scipy.stats import ttest_rel, wilcoxon

    a = df_a.groupby("item_id", sort=False)["correct"].mean()
    b = df_b.groupby("item_id", sort=False)["correct"].mean()
    공통 = a.index.intersection(b.index)
    if len(공통) < 2:
        return {"error": "공통 문항이 2개 미만입니다."}
    a, b = a.loc[공통], b.loc[공통]

    # --- McNemar (문항별 다수결을 정답/오답으로) ---
    a_bin, b_bin = (a > 0.5), (b > 0.5)
    n01 = int((~a_bin & b_bin).sum())     # A만 틀리고 B만 맞음
    n10 = int((a_bin & ~b_bin).sum())     # A만 맞고 B만 틀림
    if n01 + n10 == 0:
        mcnemar_p = 1.0
    else:
        from scipy.stats import binomtest

        mcnemar_p = float(binomtest(n10, n01 + n10, 0.5).pvalue)

    # --- 문항별 정확도 차이 ---
    diff = (b - a).values
    if len(diff) > 1 and diff.std(ddof=1) > 0:
        t_res = ttest_rel(b, a)
        t_p = float(t_res.pvalue)
        try:
            w_p = float(wilcoxon(b, a).pvalue)
        except Exception:
            w_p = float("nan")
    else:
        t_p = w_p = 1.0

    return {
        "n_items": len(공통),
        f"{name_a} 정확도": round(float(a.mean()), 4),
        f"{name_b} 정확도": round(float(b.mean()), 4),
        "차이(B-A)": round(float(b.mean() - a.mean()), 4),
        "McNemar_불일치": f"A만맞음 {n10} / B만맞음 {n01}",
        "McNemar_p": round(mcnemar_p, 4),
        "대응t_p": round(t_p, 4),
        "Wilcoxon_p": round(w_p, 4) if not math.isnan(w_p) else None,
        "유의(alpha=%.2f)" % alpha: bool(min(mcnemar_p, t_p) < alpha),
        "주의": "짝지은 자료다. 독립표본 검정을 쓰면 검정력을 잃는다.",
    }


# --------------------------------------------------------------------------
# 표본 크기
# --------------------------------------------------------------------------
def required_n(
    p_expected: float = 0.75,
    half_width: float = 0.10,
    alpha: float = 0.05,
    *,
    icc: float = 0.0,
    repeats: int = 1,
) -> dict[str, Any]:
    """원하는 정밀도를 얻으려면 문항이 몇 개 필요한가.

    `icc` 를 주면 **군집 설계를 반영**한다.
    같은 문항을 여러 번 돌려도 실효 표본은 시행 수만큼 늘지 않는다.

        설계효과 = 1 + (repeats - 1) x icc
        실효표본 = 전체시행 / 설계효과

    이것이 이번 주의 실용적 결론이다.
    **문항 10개 x 20회보다 문항 40개 x 5회가 낫다.**
    """
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    n_simple = math.ceil(z**2 * p_expected * (1 - p_expected) / half_width**2)
    deff = 1 + (repeats - 1) * icc
    n_trials = math.ceil(n_simple * deff)
    n_items = math.ceil(n_trials / repeats)

    return {
        "가정": f"정확도 약 {p_expected:.0%}, 반폭 ±{half_width:.0%}, alpha={alpha}",
        "독립가정 시 필요 시행": n_simple,
        "설계효과": round(deff, 2),
        "군집 보정 후 필요 시행": n_trials,
        "반복": repeats,
        "필요 문항 수": n_items,
    }


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 문제 상황

```
문항 20개 × 반복 10회 = 200 시행

관측 200개?  →  아니다.
```

같은 문항을 10번 돌린 것입니다.
어떤 문항은 늘 맞고, 어떤 문항은 늘 틀립니다.
**문항 안의 관측은 서로 닮아 있습니다.**

여러분은 이 구조를 압니다. **군집자료**입니다.
학교 안의 학생, 병원 안의 환자, 가구 안의 가구원과 같습니다.

In [ ]:
import numpy as np
import pandas as pd
from evaluate import summarize, consistency

rng = np.random.default_rng(2026)

def 가짜평가(item_ps, repeats):
    """문항별 정답 확률을 주면 가짜 평가 결과를 만든다."""
    행 = []
    for i, p in enumerate(item_ps):
        for r in range(repeats):
            행.append({"item_id": f"q{i:02d}", "rep": r, "condition": "sim",
                       "correct": bool(rng.random() < p),
                       "answer": "", "elapsed": 0.0, "error": None})
    return pd.DataFrame(행)


# 두 시나리오 -- 정확도는 같지만 구조가 다르다
A = 가짜평가([0.7] * 20, 10)                    # 모든 문항이 똑같이 어렵다
B = 가짜평가([1.0] * 14 + [0.0] * 6, 10)        # 늘 맞거나 늘 틀린다

for 이름, d in [("A: 문항 난이도 동일", A), ("B: 늘맞음/늘틀림", B)]:
    s = summarize(d)
    print(f"{이름}")
    print(f"   정확도      {s['accuracy']:.2f}")
    print(f"   naive CI    [{s['naive_ci'][0]:.3f}, {s['naive_ci'][1]:.3f}]  "
          f"폭 {s['naive_ci'][1]-s['naive_ci'][0]:.3f}")
    print(f"   cluster CI  [{s['cluster_ci'][0]:.3f}, {s['cluster_ci'][1]:.3f}]  "
          f"폭 {s['cluster_ci'][1]-s['cluster_ci'][0]:.3f}")
    print(f"   ICC {s['icc']}   설계효과 {s['design_effect']}   실효표본 {s['effective_n']}")
    print()

### 무엇이 보이나

**정확도는 둘 다 0.7 입니다.** 그런데:

- **A**: ICC ≈ 0, 설계효과 ≈ 1 → naive 와 cluster 구간이 거의 같다
- **B**: ICC ≈ 1, 설계효과 ≈ 10 → **cluster 구간이 훨씬 넓다**

B에서 실효표본이 200이 아니라 20입니다.
**같은 문항을 10번 돌려 봐야 그 문항의 난이도라는 하나의 정보를 10번 얻은 것**뿐입니다.

---
## 2. ★ 시뮬레이션 — naive 구간은 정말 좁은가

말로 하지 말고 **확인**합시다. 이 수업의 방식입니다.

참값을 정해 두고 가짜 평가를 300번 돌려서,
95% 신뢰구간이 참값을 **몇 번이나 포함하는지** 셉니다.

### 먼저: 무엇을 추정하려는 것인가

```
질문 1: "이 20문항에서 우리 에이전트의 정확도는?"
        → 문항이 고정. naive 도 괜찮다.

질문 2: "이런 유형의 질문 전반에서 얼마나 할까?"
        → 문항도 표본이다.
```

**실제로 알고 싶은 것은 대부분 질문 2입니다.**
그러면 문항 자체가 모집단에서 뽑은 표본이고,
문항이 달랐다면 결과도 달랐을 것이므로 그 변동이 오차에 들어가야 합니다.

> 통계학에서 익숙한 이야기입니다. **고정효과와 임의효과의 구분**입니다.
> 대부분의 AI 평가 코드는 이 질문을 아예 하지 않습니다.

In [ ]:
참값 = 0.70
a, b = 1.4, 1.4 * (1 - 참값) / 참값       # Beta 평균이 정확히 참값
N_SIM = 300
n_items, repeats = 20, 10

naive_hit = cluster_hit = 0
naive_폭, cluster_폭 = [], []

for i in range(N_SIM):
    item_ps = rng.beta(a, b, n_items)      # 매번 문항을 새로 뽑는다
    d = 가짜평가(item_ps, repeats)
    s = summarize(d)
    naive_hit += s["naive_ci"][0] <= 참값 <= s["naive_ci"][1]
    cluster_hit += s["cluster_ci"][0] <= 참값 <= s["cluster_ci"][1]
    naive_폭.append(s["naive_ci"][1] - s["naive_ci"][0])
    cluster_폭.append(s["cluster_ci"][1] - s["cluster_ci"][0])
    if (i+1) % 50 == 0:
        print(".", end="", flush=True)
print(" 완료\n")

print(f"참값 {참값}, 문항 {n_items} x 반복 {repeats}, {N_SIM}회 시뮬레이션")
print(f"  naive   95% 구간 커버리지 : {naive_hit/N_SIM:6.1%}   평균 폭 {np.mean(naive_폭):.3f}")
print(f"  cluster 95% 구간 커버리지 : {cluster_hit/N_SIM:6.1%}   평균 폭 {np.mean(cluster_폭):.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["naive", "cluster"], [naive_hit/N_SIM, cluster_hit/N_SIM],
            color=["#C44E52", "#4C72B0"])
axes[0].axhline(0.95, ls="--", c="black", label="nominal 95%")
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("coverage")
axes[0].set_title("CI coverage (target 95%)"); axes[0].legend()
for i, v in enumerate([naive_hit/N_SIM, cluster_hit/N_SIM]):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center")

axes[1].hist(naive_폭, bins=25, alpha=0.6, label="naive", color="#C44E52")
axes[1].hist(cluster_폭, bins=25, alpha=0.6, label="cluster", color="#4C72B0")
axes[1].set_xlabel("CI width"); axes[1].set_ylabel("count")
axes[1].set_title("CI width distribution"); axes[1].legend()

plt.tight_layout(); plt.show()

### 결론

**naive 구간은 실제보다 훨씬 좁습니다.** 목표 95%인데 68% 안팎밖에 못 잡습니다.

**이것이 왜 위험한가**

구간이 좁으면 **없는 차이를 있다고 말하게 됩니다.**
"프롬프트 A가 B보다 낫다"고 논문에 쓰는데 사실은 우연일 수 있습니다.

> 여러분은 이 문제를 아는 사람들입니다. 제대로 하십시오.

---
## 3. 어떻게 보정하는가

가장 단순한 방법: **문항별 정확도를 하나의 관측으로 본다.**

```python
문항정확도 = df.groupby("item_id")["correct"].mean()   # 20개
추정치 = 문항정확도.mean()
표준오차 = 문항정확도.std(ddof=1) / sqrt(20)
구간 = 추정치 ± t(19, .975) × 표준오차
```

표본이 200이 아니라 **20**입니다.

In [ ]:
# 직접 계산해서 summarize() 와 맞는지 확인
from scipy.stats import t as t_dist

d = 가짜평가(rng.beta(a, b, 20), 10)
문항정확도 = d.groupby("item_id")["correct"].mean()
m = len(문항정확도)
추정치 = 문항정확도.mean()
se = 문항정확도.std(ddof=1) / np.sqrt(m)
tcrit = t_dist.ppf(0.975, df=m-1)
직접 = (추정치 - tcrit*se, 추정치 + tcrit*se)

s = summarize(d)
print(f"직접 계산     : [{직접[0]:.4f}, {직접[1]:.4f}]")
print(f"summarize()  : [{s['cluster_ci'][0]:.4f}, {s['cluster_ci'][1]:.4f}]")
print(f"일치: {abs(직접[0]-s['cluster_ci'][0]) < 0.001}")

---
## 4. ICC 와 설계효과

```
ICC = 문항 안의 관측이 얼마나 닮았는가  (0 ~ 1)

설계효과 = 1 + (반복수 - 1) × ICC
실효표본 = 전체시행 / 설계효과
```

In [ ]:
행 = []
for icc목표, ps in [
    ("0에 가까움", [0.7]*20),
    ("중간",      list(rng.beta(3, 1.3, 20))),
    ("1에 가까움", [1.0]*14 + [0.0]*6),
]:
    d = 가짜평가(ps, 10)
    s = summarize(d)
    행.append({"시나리오": icc목표, "정확도": s["accuracy"], "ICC": s["icc"],
              "설계효과": s["design_effect"], "전체시행": s["n_trials"],
              "실효표본": s["effective_n"],
              "naive폭": round(s["naive_ci"][1]-s["naive_ci"][0], 3),
              "cluster폭": round(s["cluster_ci"][1]-s["cluster_ci"][0], 3)})
pd.DataFrame(행)

---
## 5. ★ 실용적 결론 — 문항을 늘려라

In [ ]:
from evaluate import required_n

print("목표: 정확도를 ±10%p 로 추정 (정확도 약 75% 예상)\n")
for icc in [0.0, 0.3, 0.5, 0.7, 0.9]:
    r = required_n(0.75, 0.10, icc=icc, repeats=5)
    print(f"  ICC={icc}: 설계효과 {r['설계효과']:4.1f}  "
          f"필요 시행 {r['군집 보정 후 필요 시행']:4d}  필요 문항 {r['필요 문항 수']:3d}개")

In [ ]:
# 같은 예산(총 시행 수)이라면 문항과 반복을 어떻게 나눌까?
print("ICC = 0.7 일 때, 총 시행 200회를 어떻게 나눌 것인가\n")
print(f"{'문항':>5s} {'반복':>5s} {'총시행':>7s} {'설계효과':>8s} {'실효표본':>8s}")
for 문항, 반복 in [(10, 20), (20, 10), (40, 5), (67, 3)]:
    deff = 1 + (반복 - 1) * 0.7
    총 = 문항 * 반복
    print(f"{문항:5d} {반복:5d} {총:7d} {deff:8.1f} {총/deff:8.1f}")

print()
print("-> 같은 200회라도 문항을 늘릴수록 실효표본이 커집니다.")
print("   **반복은 3~5회면 충분합니다.**")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
반복들 = np.arange(1, 21)
for icc, c in [(0.2, "#4C72B0"), (0.5, "#DD8452"), (0.8, "#C44E52")]:
    실효 = 200 / (1 + (반복들 - 1) * icc)
    ax.plot(반복들, 실효, "o-", color=c, label=f"ICC={icc}", markersize=4)
ax.set_xlabel("repeats per item  (total trials fixed at 200)")
ax.set_ylabel("effective sample size")
ax.set_title("More repeats does NOT mean more information")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **에이전트 평가에서 사람들이 흔히 하는 실수**
>
> 문항은 적게 만들고 반복을 많이 돌립니다. 문항 만들기가 귀찮으니까요.
>
> 그런데 ICC가 높으면 반복을 늘려도 실효표본이 거의 안 늘어납니다.
> **문항을 늘리십시오.**

---
## 6. 정확도 ≠ 일관성

In [ ]:
A2 = 가짜평가([0.5] * 20, 10)                 # 모든 문항을 반반씩
B2 = 가짜평가([1.0] * 10 + [0.0] * 10, 10)     # 절반은 늘 맞고 절반은 늘 틀림

for 이름, d in [("A: 매번 운", A2), ("B: 못 푸는 게 정해짐", B2)]:
    s = summarize(d)
    c = consistency(d)
    print(f"{이름}")
    print(f"   정확도 {s['accuracy']:.2f}   ICC {s['icc']}")
    print(f"   {c}")
    print()

print("정확도가 같습니다. 성질은 완전히 다릅니다.")
print()
print("  A -> 온도를 낮추고, 출력을 구조화하고, 다수결을 쓴다")
print("  B -> 어떤 유형을 못 하는지 찾아 도구를 추가하거나 프롬프트를 고친다")
print()
print("**고쳐야 할 것이 완전히 다릅니다.**")

> 실무적으로 중요한 점이 하나 더 있습니다.
> **A 같은 에이전트는 사용자 신뢰를 잃습니다.**
> 어제 되던 것이 오늘 안 되면 사람들이 도구를 안 씁니다.
> 평균 정확도가 같아도 그렇습니다.

---
## 연습문제

**문제 1.** 혼합효과 로지스틱 모형으로 정확도를 추정하시오.
문항을 임의절편으로 넣고, 군집 평균 방법과 결과를 비교하시오.

**문제 2.** 문항 수를 5, 10, 20, 40 으로 바꿔 가며 커버리지 시뮬레이션을 하시오.
문항이 적으면 cluster 구간도 부정확한가?

**문제 3.** ICC 추정량 자체의 변동을 보시오.
같은 조건으로 100번 시뮬레이션해 ICC 추정치의 분포를 그리시오.

**문제 4.** 10주차의 검색 평가(Recall@3, n=15)에 군집 보정이 필요한가?
필요하다면 무엇이 군집인가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
```python
import statsmodels.formula.api as smf
d["y"] = d["correct"].astype(int)
m = smf.mixedlm("y ~ 1", d, groups=d["item_id"]).fit()
print(m.summary())
```
엄밀히는 이분 결과에 선형혼합모형을 쓰는 것이 부적절하고 GLMM 이 맞습니다
(`statsmodels` 의 `BinomialBayesMixedGLM` 또는 R 의 `lme4::glmer`).
다만 **군집 평균 방법과 결과가 크게 다르지 않습니다.**

군집 평균 방법의 장점: **가정이 적고 설명하기 쉽습니다.**
논문에 "문항을 군집으로 보아 문항별 정확도의 표본평균과 그 표준오차로
구간을 구했다" 라고 쓰면 심사자가 이해합니다.

**문제 2.** 문항이 5개면 t분포 자유도가 4라 구간이 매우 넓어집니다.
커버리지는 유지되지만 **쓸모없이 넓습니다.**
→ 문항은 최소 15~20개는 되어야 합니다.

**문제 4.** 필요합니다. 다만 군집이 다릅니다.
검색 평가는 문항당 1회만 실행했으므로(결정적 알고리즘) 반복에 의한 군집은 없습니다.
그러나 **문항 자체가 표본**이라는 문제는 그대로입니다.
n=15 의 Wilson 구간은 "이 15문항에서의" 성능이고,
"새 질문 전반"으로 일반화하려면 문항 표본의 대표성을 따져야 합니다.

</details>

---
## 정리

- 반복 실행은 **군집자료**다. 관측이 독립이 아니다
- **naive 구간은 실제보다 좁다** — 시뮬레이션으로 확인했다 (68% vs 95%)
- 무엇을 추정하려는지 정하라: **이 문항들 vs 이런 유형 전반**
- **설계효과 = 1 + (반복−1) × ICC.** 실효표본이 그만큼 준다
- **문항을 늘려라.** 반복은 3~5회면 충분
- **정확도 ≠ 일관성.** 고쳐야 할 것이 다르다

**다음 노트북** → [`03_ab_test.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week11/03_ab_test.ipynb)